# Lecture 27 - LLM Deployment

[![View notebook on Github](https://img.shields.io/static/v1.svg?logo=github&label=Repo&message=View%20On%20Github&color=lightgrey)](https://github.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/blob/main/Lectures/Theme_4-Deployment/Lecture_27-LLM_Deployment/Lecture_27-LLM_Deployment.ipynb)
[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avakanski/Fall-2026-Applied-Data-Science-with-Python/blob/main/Lectures/Theme_4-Deployment/Lecture_27-LLM_Deployment/Lecture_27-LLM_Deployment.ipynb)
[![View on Website](https://img.shields.io/static/v1.svg?logo=quarto&label=Web&message=View%20on%20Website&color=blue)](https://avakanski.github.io/Fall-2026-Applied-Data-Science-with-Python/Lectures/Theme_4-Deployment/Lecture_27-LLM_Deployment/Lecture_27-LLM_Deployment.html)

<a id='top'></a>

- [27.1 Introduction to LLM Deployment](#27.1-introduction-to-llm-deployment)
    - [27.1.1 Deployment Options](#27.1.1-deployment-options)
- [27.2 LLM Inference Basics](#27.2-llm-inference-basics)
    - [27.2.1 The KV Cache](#27.2.1-the-kv-cache)
    - [27.2.2 Performance Metrics](#27.2.2-performance-metrics)
- [27.3 Inference Optimization](#27.3-inference-optimization)
    - [27.3.1 Quantization](#27.3.1-quantization)
    - [27.3.2 Paged Attention and Continuous Batching](#27.3.2-paged-attention-and-continuous-batching)
    - [27.3.3 Speculative Decoding](#27.3.3-speculative-decoding)
    - [27.3.4 Other Optimization Techniques](#27.3.4-other-optimization-techniques)
- [27.4 LLM Serving Frameworks](#27.4-llm-serving-frameworks)
    - [27.4.1 Offline Batch Inference with vLLM](#27.4.1-offline-batch-inference-with-vllm)
    - [27.4.2 Serving an OpenAI-Compatible API](#27.4.2-serving-an-openai-compatible-api)
    - [27.4.3 Running LLMs Locally with Ollama](#27.4.3-running-llms-locally-with-ollama)
- [27.5 Evaluation Pipelines](#27.5-evaluation-pipelines)
    - [27.5.1 Offline and Online Evaluation](#27.5.1-offline-and-online-evaluation)
    - [27.5.2 Building an Evaluation Pipeline](#27.5.2-building-an-evaluation-pipeline)
- [27.6 Monitoring and Production Considerations](#27.6-monitoring-and-production-considerations)
- [References](#references)

<a id="27.1-introduction-to-llm-deployment"></a>

## 27.1 Introduction to LLM Deployment

In Lecture 16, we deployed machine learning models as web applications with Flask, Django, and FastAPI, and in Lecture 17 we packaged applications in Docker containers. The same principles apply to Large Language Models, but the size of LLMs and the way they generate text create new challenges:

- **Memory**: An LLM with 8 billion parameters requires 16 GB of GPU memory just to store its weights in 16-bit precision, and a 70-billion-parameter model requires 140 GB, which exceeds the memory of any single GPU.
- **Latency**: LLMs generate text one token at a time, and each token requires a full forward pass through the network. Users expect fast response from LLMs.
- **Throughput and cost**: GPUs are expensive, and to keep the cost per request low a deployed LLM must serve many users at the same time.
- **Quality assurance**: The outputs of LLMs are open-ended text, which makes it difficult to verify that a model, a prompt, or a configuration change did not reduce the quality of the responses.

In this lecture, we will first study how LLM inference works and how to measure its performance. Next, we will review the main inference optimization techniques, and use the vLLM serving engine to deploy a model with an OpenAI-compatible API. Lastly, we will build an evaluation pipeline for assessing the quality of an LLM application.

<a id="27.1.1-deployment-options"></a>

### 27.1.1 Deployment Options

There are three main options for using LLMs in an application.

- **Proprietary model APIs**: models such as GPT, Claude, and Gemini are accessed through the provider's API, and the provider handles all infrastructure. This is the fastest way to start, and gives access to the most capable models. The drawbacks are the per-token cost, the need to send data to a third party, and limited control over model versions.
- **Managed inference for open models**: Services such as Hugging Face Inference Providers, Together AI, Fireworks AI, Amazon Bedrock, and Azure AI Foundry host open-weight models (like Llama, Qwen, Mistral, DeepSeek, Gemma) behind an API. We used Hugging Face Inference Providers in Lectures 24 and 25.
- **Self-hosting**: The organization runs an open-weight model on its own GPUs, either in the cloud or on-premises. This provides full control over data privacy, latency, customization, and can be cheaper at high utilization. However, it requires expertise in inference optimization and GPU infrastructure, which is the main topic of this lecture.

Many organizations use a combination of these options, e.g., a proprietary API for complex reasoning tasks, and a self-hosted small model for high-volume, simple tasks such as classification or extraction.

<a id="27.2-llm-inference-basics"></a>

## 27.2 LLM Inference Basics

As we know, decoder-only Transformer LLMs generate text *autoregressively*, i.e., the model predicts the next token, appends it to the input, and repeats the process. Generating a response for a prompt consists of two distinct phases.

- **Prefill phase**: the tokens of the input prompt are processed in a single forward pass. Because the prompt tokens are processed in parallel, this phase makes full use of the GPU's computational power, and it is bounded by the available compute. The prefill phase produces the first output token.
- **Decode phase**: the output tokens are generated one at a time, each requiring a forward pass. In each pass, the model computes only one token, but it must read all model weights from GPU memory. Therefore, this phase is bounded by the memory bandwidth. That is, the speed is limited by how fast the weights can be transferred from memory to the compute units.

Most of the optimization techniques covered next either reduce the amount of memory that must be read per token, or increase the amount of useful work done for each read of the weights.

<a id="27.2.1-the-kv-cache"></a>

### 27.2.1 The KV Cache

In the self-attention layers, each new token attends to all previous tokens through their key and value vectors. Without any caching, generating the 500th token would require recomputing the keys and values of all 499 previous tokens, and the cost of generation would grow quadratically with the length of the sequence.

The **KV cache** stores the key and value vectors of all previous tokens in GPU memory, so that at each decoding step only the keys and values of the newest token are computed. The KV cache is used by all modern inference engines, and it makes generation much faster. However, it consumes a large amount of GPU memory, which grows linearly with the sequence length and with the batch size (the number of sequences processed concurrently).

The size of the KV cache in bytes is:

$$
\text{KV cache size} = 2 \times n_{\text{layers}} \times n_{\text{kv heads}} \times d_{\text{head}} \times L \times B \times b
$$

where the factor 2 accounts for both keys and values, $n_{\text{layers}}$, $n_{\text{kv heads}}$, and $d_{\text{head}}$ are self-explanatory, $L$ is the sequence length, $B$ is the batch size, and $b$ is the number of bytes per value (e.g., 2 bytes for 16-bit precision). Recall that modern LLMs use *grouped-query attention (GQA)*, where several query heads share the same key and value heads, which makes $n_{\text{kv heads}}$ smaller than the number of attention heads and reduces the cache size.

The next cell calculates the memory requirements for two models from the Llama 3.1 family.

In [ ]:
def weights_memory_gb(n_params_billion, bytes_per_param=2):
    # Memory for model weights (2 bytes per parameter in FP16/BF16)
    return n_params_billion * 1e9 * bytes_per_param / 1e9

def kv_cache_gb(n_layers, n_kv_heads, head_dim, seq_len, batch_size, bytes_per_value=2):
    # Memory for the KV cache: keys and values for every layer, head, and token
    return 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * bytes_per_value / 1e9

# Architecture parameters from the models' config.json files
models = {
    "Llama-3.1-8B":  {"params": 8,  "n_layers": 32, "n_kv_heads": 8, "head_dim": 128},
    "Llama-3.1-70B": {"params": 70, "n_layers": 80, "n_kv_heads": 8, "head_dim": 128},
}

for name, cfg in models.items():
    per_token_kb = kv_cache_gb(cfg["n_layers"], cfg["n_kv_heads"], cfg["head_dim"], 1, 1) * 1e6
    print(f"{name}: weights = {weights_memory_gb(cfg['params']):.0f} GB, "
          f"KV cache per token = {per_token_kb:.0f} KB")
    for seq_len, batch in [(8192, 1), (8192, 32)]:
        kv = kv_cache_gb(cfg["n_layers"], cfg["n_kv_heads"], cfg["head_dim"], seq_len, batch)
        print(f"   {batch:>2} sequence(s) x {seq_len} tokens -> KV cache = {kv:.1f} GB")

Llama-3.1-8B: weights = 16 GB, KV cache per token = 131 KB
    1 sequence(s) x 8192 tokens -> KV cache = 1.1 GB
   32 sequence(s) x 8192 tokens -> KV cache = 34.4 GB
Llama-3.1-70B: weights = 140 GB, KV cache per token = 328 KB
    1 sequence(s) x 8192 tokens -> KV cache = 2.7 GB
   32 sequence(s) x 8192 tokens -> KV cache = 85.9 GB


For the 8B model, serving a batch of 32 sequences with 8,192 tokens each requires more memory for the KV cache (about 34 GB) than for the model weights (16 GB). This shows that the KV cache is often the main factor that limits how many users a GPU can serve at the same time.

<a id="27.2.2-performance-metrics"></a>

### 27.2.2 Performance Metrics

The performance of an LLM deployment is described with the following metrics:

- **Time to First Token (TTFT)**: the time between sending a request and receiving the first output token. It is dominated by the prefill phase and by the time the request waits in a queue. It determines how responsive an application is in interactive use, such as a chatbot.
- **Time Per Output Token (TPOT)**, also called Inter-Token Latency (ITL): the average time to generate each next token in the decode phase. For reference, a typical person reads at about 4 to 5 words per second, so a TPOT below 100 ms gives a smooth reading experience.
- **End-to-end latency**: the total time to generate the full response, approximately equal to TTFT plus TPOT multiplied by the number of output tokens.
- **Throughput**: the total number of output tokens per second that the system generates across all users. Throughput determines the cost per token.

There is a trade-off between latency and throughput: processing more requests in a batch increases throughput, but each individual request may be slower. Production deployments usually define *Service Level Objectives (SLOs)*, such as "TTFT below 500 ms for 95% of requests", and maximize throughput under those constraints.

In the next cells, we load `Qwen2.5-0.5B-Instruct` model and measure these metrics. The `TextIteratorStreamer` returns the generated text piece by piece while generation runs in a separate thread, which allows us to record the time when the first token arrives.

In [ ]:
import time, gc
from threading import Thread
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

# Hide non-critical generation warnings
transformers.logging.set_verbosity_error()

# Load the model and tokenizer
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def build_inputs(prompt, model):
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)

def measure_latency(model, prompt, max_new_tokens=200):
    inputs = build_inputs(prompt, model)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer)

    start = time.perf_counter()
    thread = Thread(target=model.generate, kwargs=kwargs)
    thread.start()

    first_token_time, pieces = None, []
    for text in streamer:
        if first_token_time is None and text:
            first_token_time = time.perf_counter()
        pieces.append(text)
    thread.join()
    end = time.perf_counter()

    output = "".join(pieces)
    n_tokens = len(tokenizer(output)["input_ids"])
    return {"prompt_tokens": inputs["input_ids"].shape[1],
            "output_tokens": n_tokens,
            "TTFT (ms)": round(1000 * (first_token_time - start), 1),
            "TPOT (ms)": round(1000 * (end - first_token_time) / max(n_tokens - 1, 1), 1),
            "end-to-end (s)": round(end - start, 2),
            "tokens/s": round(n_tokens / (end - start), 1)}

# Warm-up run (the first call includes one-time CUDA initialization costs)
_ = measure_latency(model, "Hello!", max_new_tokens=10)

measure_latency(model, "Explain what a KV cache is in a Transformer model.")

{'prompt_tokens': 41,
 'output_tokens': 200,
 'TTFT (ms)': 107.0,
 'TPOT (ms)': 44.8,
 'end-to-end (s)': 9.03,
 'tokens/s': 22.2}

Note that the streamer returns text in chunks of whole words rather than individual tokens, so the measured values are approximations. Benchmarking tools for inference servers, such as `vllm bench` and GuideLLM, measure these metrics precisely under realistic loads with many concurrent users.

Next, we observe the effect of the prompt length on TTFT. Since individual timings vary from run to run, we repeat each measurement three times and report the median values. A long prompt increases the time of the prefill phase, while the TPOT remains almost unchanged.

In [ ]:
import statistics

def median_latency(model, prompt, max_new_tokens=100, n_runs=3):
    runs = [measure_latency(model, prompt, max_new_tokens) for _ in range(n_runs)]
    return {k: statistics.median(r[k] for r in runs) for k in runs[0]}

short_prompt = "Summarize the benefits of unit testing."
long_prompt = ("Summarize the following notes about software testing.\n" +
               "Unit tests check small pieces of code in isolation, and they help catch bugs early. " * 150)

for name, p in [("short", short_prompt), ("long", long_prompt)]:
    print(name, median_latency(model, p))

short {'prompt_tokens': 38, 'output_tokens': 100, 'TTFT (ms)': 59.7, 'TPOT (ms)': 27.3, 'end-to-end (s)': 2.77, 'tokens/s': 36.1}
long {'prompt_tokens': 2590, 'output_tokens': 100, 'TTFT (ms)': 169.4, 'TPOT (ms)': 26.6, 'end-to-end (s)': 2.8, 'tokens/s': 35.7}


**The Effect of the KV Cache**

To see the effect of the KV cache, we can disable it with the argument `use_cache=False`. In this case, the model recomputes the keys and values for all previous tokens at every decoding step, and the generation becomes noticeably slower, especially for longer sequences.

For a short prompt and a small model such as ours, the difference can be small, because recomputing a few hundred tokens costs almost nothing. Therefore, we will use the long prompt from the previous cell. Without the cache, every decoding step must reprocess all of the approximately 2,600 prompt tokens, which is similar in cost to repeating the prefill phase for each new token.

In [ ]:
inputs = build_inputs(long_prompt, model)
n_new = 100

for use_cache in [True, False]:
    torch.cuda.synchronize()
    start = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=n_new, min_new_tokens=n_new,
                         do_sample=False, use_cache=use_cache)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    print(f"use_cache={use_cache}: {elapsed:.2f} s for {n_new} tokens ({n_new / elapsed:.1f} tokens/s)")

use_cache=True: 3.44 s for 100 tokens (29.0 tokens/s)
use_cache=False: 14.97 s for 100 tokens (6.7 tokens/s)


<a id="27.3-inference-optimization"></a>

## 27.3 Inference Optimization

This section reviews the techniques for making LLM inference faster and cheaper. Some of them change the model (quantization), some change the decoding algorithm (speculative decoding), and others change how the serving system manages memory and requests (paged attention and continuous batching).

<a id="27.3.1-quantization"></a>

### 27.3.1 Quantization

We already covered *quantization* and we explained that it reduces the numerical precision of the model weights. A model quantized to 4 bits needs approximately four times less memory for its weights than in 16-bit precision. This allows larger models to fit on smaller GPUs, and because the decode phase is bounded by memory-bandwidth, reading fewer bytes per token can also make generation faster.

Common quantization methods include:

- **bitsandbytes**: Quantizes the weights to 8 or 4 bits (NF4 format). It is widely used for fine-tuning with QLoRA, and for running large models on limited hardware.
- **GPTQ and AWQ**: Post-training quantization methods that use a small calibration dataset to minimize the quantization error. AWQ (Activation-aware Weight Quantization) protects the small fraction of weights that are most important for the model's activations. Many pre-quantized GPTQ and AWQ models are available on the Hugging Face Hub, and they are supported by serving engines with fast GPU kernels.
- **FP8**: An 8-bit floating point format supported in hardware by recent NVIDIA GPUs (Hopper and Blackwell architectures). FP8 quantization typically causes a negligible loss of accuracy and is commonly used in production.
- **GGUF**: The file format used by `llama.cpp` and Ollama, which supports various quantization levels (from 2 to 8 bits) and is designed for running LLMs on CPUs and laptops.

Quantization involves a trade-off, since lower precision reduces memory and cost, but it can reduce the quality of the model.

In the next example, we load the `Qwen2.5-1.5B-Instruct` model in 16-bit precision and in 4-bit precision with `bitsandbytes`, and compare their memory footprints and outputs.

In [ ]:
!pip install -q bitsandbytes

In [ ]:
from transformers import BitsAndBytesConfig

model_id_15 = "Qwen/Qwen2.5-1.5B-Instruct"

model_fp16 = AutoModelForCausalLM.from_pretrained(model_id_15, dtype=torch.float16, device_map="cuda")

bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_quant_type="nf4",            # 4-bit NormalFloat format
                                bnb_4bit_compute_dtype=torch.float16)  # computations in FP16
model_4bit = AutoModelForCausalLM.from_pretrained(model_id_15, quantization_config=bnb_config, device_map="cuda")

print(f"FP16 model:  {model_fp16.get_memory_footprint() / 1e9:.2f} GB")
print(f"4-bit model: {model_4bit.get_memory_footprint() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FP16 model:  3.09 GB
4-bit model: 1.12 GB


In [ ]:
prompt = "List three advantages of using Docker containers for deploying machine learning models."

for name, m in [("FP16", model_fp16), ("4-bit", model_4bit)]:
    inputs = build_inputs(prompt, m)
    out = m.generate(**inputs, max_new_tokens=120, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"=== {name} ===\n{text}\n")
    print(measure_latency(m, prompt, max_new_tokens=120), "\n")

=== FP16 ===
Docker containers offer several advantages when it comes to deploying machine learning (ML) models:

1. **Portability and Isolation**: Containers provide an isolated environment that can run the same software on different machines or even in different cloud environments without requiring any changes. This is particularly useful for ML models where dependencies might vary between development, testing, and production environments.

2. **Resource Efficiency**: By packaging all necessary components into a single container, including libraries, dependencies, and configurations, Docker reduces the overhead associated with managing multiple processes and their resources. This leads to more efficient use of system resources, especially

{'prompt_tokens': 42, 'output_tokens': 120, 'TTFT (ms)': 259.4, 'TPOT (ms)': 63.8, 'end-to-end (s)': 7.85, 'tokens/s': 15.3} 

=== 4-bit ===
1. **Portability**: Docker containers allow you to package your machine learning (ML) model along with its 

The 4-bit model uses about one third of the memory of the FP16 model, and its responses are similar in quality. The used memory is not exactly one quarter, since some layers such as the embeddings are kept in higher precision. Note however that the 4-bit model in this example is not faster. This is because `bitsandbytes` dequantizes the weights back to 16 bits during every forward pass, which adds computational overhead. Serving engines such as vLLM use specialized GPU kernels for AWQ, GPTQ, and FP8 models that convert the memory savings into speed improvements.

<a id="27.3.2-paged-attention-and-continuous-batching"></a>

### 27.3.2 Paged Attention and Continuous Batching

**Paged Attention**

Before 2023, inference systems reserved a contiguous block of GPU memory for the KV cache of each request, sized for the maximum possible output length. Because most responses are shorter than the maximum length, a large portion of the reserved memory was wasted, and studies reported that 60 to 80% of the KV cache memory was unused.

*Paged Attention* was introduced with the vLLM serving engine, and it borrows the idea of virtual memory paging from operating systems. The KV cache is divided into small fixed-size blocks (e.g., 16 tokens each), which do not need to be contiguous in memory, and blocks are allocated on demand as the sequence grows. This reduces memory waste, which allows the system to serve several times more concurrent requests on the same GPU. Paged memory also enables *prefix caching*, which means that when many requests share the same beginning (e.g., the same long system prompt), the KV cache blocks for the shared prefix are computed once and reused.

**Continuous Batching**

Batching several sequences together improves GPU utilization, because the model weights are read from memory once per decoding step and used for all sequences in the batch. With traditional *static batching*, a batch of sequences is processed until the longest sequence in the batch is finished, and new requests must wait until the whole batch completes. Since output lengths vary greatly, many GPU slots remain idle.

*Continuous batching* (also called in-flight batching) schedules at the level of individual decoding steps, and as soon as a request finishes, it leaves the batch, and a waiting request takes its place in the next step. Combined with paged attention, continuous batching can increase throughput by an order of magnitude compared to processing requests one at a time, and it is the core technique behind all modern LLM serving engines.

<a id="27.3.3-speculative-decoding"></a>

### 27.3.3 Speculative Decoding

Since the decode phase is limited by memory bandwidth rather than computation, the GPU has spare computational capacity while generating each token. **Speculative decoding** uses this spare capacity as follows:

1. A small, fast *draft model* generates several candidate tokens, e.g., 5 tokens.
2. The large *target model* verifies all candidate tokens in a single forward pass, in parallel, similarly to the prefill phase.
3. The candidate tokens are accepted until the first token where the target model disagrees. At that position, the target model's own prediction is used instead.

If the draft model guesses correctly most of the time, several tokens are produced for the cost of one forward pass of the large model. Importantly, the verification step guarantees that the output is the same as the output that the target model would have generated on its own, so the speed-up does not reduce quality.

Variants of speculative decoding include *prompt lookup decoding*, which uses n-grams from the prompt as candidates (effective for tasks like summarization and code editing, where the output repeats parts of the input), and methods such as Medusa and EAGLE, which add lightweight prediction heads to the target model instead of using a separate draft model.

In Hugging Face Transformers, speculative decoding (called *assisted generation*) is enabled by passing the draft model with the `assistant_model` argument. The draft and target models must use the same tokenizer, so we use `Qwen2.5-0.5B-Instruct` as the draft model for `Qwen2.5-1.5B-Instruct`.

In [ ]:
# Free the 4-bit model to save GPU memory
del model_4bit
gc.collect(); torch.cuda.empty_cache()

target_model, draft_model = model_fp16, model   # 1.5B target, 0.5B draft
inputs = build_inputs("Write a Python function that checks whether a string is a palindrome, "
                      "and explain how it works.", target_model)

def timed_generate(max_new_tokens=300, **kwargs):
    torch.cuda.synchronize()
    start = time.perf_counter()
    out = target_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, **kwargs)
    torch.cuda.synchronize()
    return out, time.perf_counter() - start

out_regular, t_regular = timed_generate()
out_spec, t_spec = timed_generate(assistant_model=draft_model)
out_lookup, t_lookup = timed_generate(prompt_lookup_num_tokens=10)

print(f"Regular decoding:       {t_regular:.2f} s")
print(f"Speculative decoding:   {t_spec:.2f} s  (speed-up {t_regular / t_spec:.2f}x)")
print(f"Prompt lookup decoding: {t_lookup:.2f} s  (speed-up {t_regular / t_lookup:.2f}x)")
print("Identical outputs:", torch.equal(out_regular, out_spec))

Regular decoding:       19.41 s
Speculative decoding:   23.63 s  (speed-up 0.82x)
Prompt lookup decoding: 10.29 s  (speed-up 1.89x)
Identical outputs: True


In the above cell, the outputs of regular and speculative decoding are identical, which confirms that speculative decoding does not change the result for greedy decoding.

However, in the above cell speculative decoding did not provide a speed-up, and it was even slower than regular decoding. The reason is that both models are small: on a T4 GPU, each forward pass of a model with 0.5 or 1.5 billion parameters is dominated by fixed overheads, so the draft model costs almost as much per step as the target model, and verifying the rejected candidate tokens is wasted work. Speculative decoding pays off when the target model is much larger and slower than the draft model. For instance, for target models with 70 billion parameters, speed-ups of 2 to 3 times are common.

On the other hand, prompt lookup decoding was almost twice as fast as regular decoding in the above cell. Prompt lookup searches for candidate n-grams in both the prompt and the tokens generated so far, and the explanation generated by the model repeats parts of the code that it wrote earlier. Prompt lookup decoding is especially effective when the output copies long spans of the input. In the next example, we ask the model to rename a variable in a function and return the full code, so most of the output tokens can be taken directly from the prompt.

In [ ]:
code_prompt = """Rename the variable `x` to `count` in the following function, and return the full updated code only.

def count_words(lines):
    x = 0
    for line in lines:
        words = line.strip().split()
        for word in words:
            if word.isalpha():
                x += 1
    print("Number of words:", x)
    return x
"""
inputs = build_inputs(code_prompt, target_model)

out_regular, t_regular = timed_generate(max_new_tokens=150)
out_lookup, t_lookup = timed_generate(max_new_tokens=150, prompt_lookup_num_tokens=10)

print(f"Regular decoding:       {t_regular:.2f} s")
print(f"Prompt lookup decoding: {t_lookup:.2f} s  (speed-up {t_regular / t_lookup:.2f}x)")
print("Identical outputs:", torch.equal(out_regular, out_lookup))
print(tokenizer.decode(out_lookup[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Regular decoding:       4.71 s
Prompt lookup decoding: 2.57 s  (speed-up 1.83x)
Identical outputs: True
```python
def count_words(lines):
    count = 0
    for line in lines:
        words = line.strip().split()
        for word in words:
            if word.isalpha():
                count += 1
    print("Number of words:", count)
    return count

# Example usage:
lines = ["Hello world", "This is a test"]
print(count_words(lines))
```

In this example, the variable `x` has been renamed to `count`. The function now returns the number of words found in the given list of lines instead of printing it.


Note that the model also added an example usage that was not requested, and its final sentence is incorrect, since the function still prints the number of words. Small models often make such mistakes, which is one of the reasons why evaluating LLM outputs is important, as discussed in Section 27.5.

<a id="27.3.4-other-optimization-techniques"></a>

### 27.3.4 Other Optimization Techniques

Several additional techniques are commonly used in LLM deployment:

- **FlashAttention**: An implementation of the attention operation that minimizes the reads and writes to GPU memory by computing attention in small tiles in the fast on-chip memory. It is mathematically exact, and it speeds up both training and the prefill phase of inference. We mentioned it briefly in Lecture 20.
- **Model parallelism**: Models that do not fit on a single GPU are split across several GPUs. *Tensor parallelism* splits individual weight matrices across GPUs within a server, and *pipeline parallelism* places different layers on different GPUs or servers.
- **Disaggregated prefill and decode**: Because the prefill phase is compute-bound and the decode phase is memory-bound, large deployments run the two phases on separate groups of GPUs, which are each optimized for their workload.
- **Knowledge distillation and smaller models**: A small model trained to imitate a large model on a specific task can often match its quality at a fraction of the cost. For many applications, choosing the smallest model that meets the quality requirements is the most effective optimization.
- **Prompt caching in APIs**: Proprietary API providers offer reduced prices and latency for repeated prompt prefixes, which applies the idea of prefix caching to API-based applications.

<a id="27.4-llm-serving-frameworks"></a>

## 27.4 LLM Serving Frameworks

An LLM serving framework combines the optimizations from the previous section into a production server. The widely used frameworks include:

- **vLLM**: An open-source engine developed at UC Berkeley, which introduced PagedAttention. It supports continuous batching, quantization (AWQ, GPTQ, FP8), speculative decoding, prefix caching, multi-LoRA serving, and tensor parallelism, and it provides an OpenAI-compatible API server. It is the most widely used open-source serving engine.
- **SGLang**: A high-performance engine that introduced RadixAttention for efficient reuse of the KV cache across requests that share prefixes. It is particularly efficient for agentic and multi-turn workloads.
- **TensorRT-LLM**: NVIDIA's library that compiles models into highly optimized engines for NVIDIA GPUs. It often achieves the best performance on NVIDIA hardware, at the cost of a more complex setup. It is commonly deployed with the NVIDIA Triton Inference Server or NVIDIA Dynamo.
- **Hugging Face Text Generation Inference (TGI)**: Hugging Face's serving engine, which powered Hugging Face's inference services. In 2025 it was placed in maintenance mode, and Hugging Face now recommends vLLM or SGLang for new deployments.
- **llama.cpp and Ollama**: Engines optimized for running quantized models (in GGUF format) on CPUs, Apple Silicon, and consumer GPUs. Ollama provides a simple command-line interface and a local API, and it is popular for running LLMs on laptops and for prototyping.

In this section, we will use vLLM to serve a model. Note that vLLM requires a GPU, and the cells below were run on a Colab T4 GPU. First, install vLLM in the next cell, and then **restart the Colab session** (Runtime > Restart session) before continuing. The restart is required for two reasons. First, it releases the GPU memory used by the models in the previous sections (vLLM does not start if there is not enough free GPU memory), and second, it loads the PyTorch version installed together with vLLM.

In [1]:
!pip install -q vllm

<a id="27.4.1-offline-batch-inference-with-vllm"></a>

### 27.4.1 Offline Batch Inference with vLLM

vLLM can be used directly from Python for *offline batch inference*, i.e., for processing a large set of prompts, such as labeling a dataset or generating synthetic data. The `LLM` class loads the model, and `SamplingParams` sets the generation parameters.

The arguments of the `LLM` class in the next cell are:

- `dtype="half"`: Uses FP16 precision, because the T4 GPU does not support the BF16 format.
- `gpu_memory_utilization=0.4`: The fraction of GPU memory that vLLM will use, i.e., 40% or about 6 GB of the 15 GB on a T4 GPU. vLLM loads the model weights (about 3 GB), and allocates the remaining memory in this budget to the paged KV cache.
- `max_model_len=4096`: The maximum sequence length (prompt and output), which limits the memory needed per request.

In [2]:
import time
from vllm import LLM, SamplingParams

llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", dtype="half",
          gpu_memory_utilization=0.4, max_model_len=4096)

sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=200)

W1121 01:22:54.929000 3934 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1121 01:22:54.939000 3934 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1121 01:22:54.944000 3934 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=3934) INFO 11-21 01:23:37 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:140: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

(MainProcess pid=3934) INFO 11-21 01:24:07 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=3934) WARNING 11-21 01:24:07 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=3934) INFO 11-21 01:24:07 [model.py:2214] Using max model len 4096
(MainProcess pid=3934) INFO 11-21 01:24:07 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=3934) INFO 11-21 01:24:09 [kernel.py:517] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (4 outputs left out)

[rank0]:W1121 01:25:27.276000 4478 torch/_inductor/utils.py:1806] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=4478) INFO 11-21 01:25:31 [backends.py:421] [backbone] Compiling a graph for compile range (1, 8192) takes 10.18 s
(EngineCore pid=4478) INFO 11-21 01:25:35 [backends.py:952] collected artifacts: 29 entries, 3 artifacts, 3739463 bytes total
(EngineCore pid=4478) INFO 11-21 01:25:35 [decorators.py:752] [backbone] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/f9c594ace54411726c5e87964f2901e4936abe094d53680faab8d0209cf211a0/rank_0_0/model
(EngineCore pid=4478) INFO 11-21 01:25:35 [monitor.py:62] [backbone] torch.compile took 23.49 s in total
(EngineCore pid=4478) INFO 11-21 01:25:35 [monitor.py:91] [backbone] Initial profiling/warmup run took 0.12 s
(EngineCore pid=4478) INFO 11-21 01:25:47 [gpu_worker.py:795] Available KV cache memory: 2.14 GiB
(EngineCore pid=4478) INFO 11-21 01:25:47 [kernel_warmup.py:294] JIT kernel warmup starting.
(EngineCore pid=4478) INFO 11-21 01:25:47 [kernel_warmup.py:304] JIT kernel warmup finished in 0.

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 18.62it/s]


(EngineCore pid=4478) INFO 11-21 01:26:27 [model_runner.py:1216] Graph capturing finished in 8 secs, took 0.41 GiB
(EngineCore pid=4478) INFO 11-21 01:26:27 [gpu_worker.py:1175] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.4, 5.83 GiB). Actual usage is 3.22 GiB for consumed memory (weights + non-torch), 0.46 GiB for peak activation, and 0.41 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=1697719604` (1.58 GiB) to fit into requested memory, or `--kv-cache-memory=10919604736` (10.17 GiB) to fully utilize gpu memory. Current kv cache memory in use is 2.14 GiB.
(EngineCore pid=4478) INFO 11-21 01:26:29 [jit_monitor.py:87] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=4478) INFO 11-21 01:26:29 [extensible_kv_cache.py:693] Memory after warmup: 10.72 GiB free, 0.05 GiB committed to the KV cache, 0.65 GiB kept for transient activations (0.48 GiB a

When the model is loaded, vLLM logs the amount of memory available for the KV cache, and the maximum number of tokens that can be cached, which determines the maximum concurrency.

Next, we generate responses for 64 prompts. vLLM schedules all prompts using continuous batching, and processes them in parallel.

In [3]:
topics = ["linear regression", "decision trees", "k-means clustering", "cross-validation",
          "gradient boosting", "convolutional networks", "word embeddings", "attention",
          "dropout", "batch normalization", "principal component analysis", "random forests",
          "logistic regression", "support vector machines", "transfer learning", "data augmentation"]
styles = ["for a high school student", "in one paragraph", "with an analogy", "with a Python example"]
prompts = [f"Explain {t} {s}." for t in topics for s in styles]
conversations = [[{"role": "user", "content": p}] for p in prompts]
print(len(conversations), "prompts")

start = time.perf_counter()
outputs = llm.chat(conversations, sampling_params)
elapsed = time.perf_counter() - start

n_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
print(f"Generated {n_tokens} tokens in {elapsed:.1f} s -> throughput {n_tokens / elapsed:.0f} tokens/s")
print("\nExample response:\n", outputs[0].outputs[0].text)

64 prompts


Rendering conversations:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 64/64 [00:20<00:00,  3.17it/s, est. speed input: 120.29 toks/s, output: 554.17 toks/s]

Generated 11186 tokens in 20.4 s -> throughput 548 tokens/s

Example response:
 Linear regression is a statistical method that is commonly used to model the relationship between a dependent variable and one or more independent variables. The goal of linear regression is to find the best-fitting straight line, called the regression line, that can predict the value of the dependent variable based on the independent variables.

In linear regression, the dependent variable is typically denoted as y, and the independent variables are denoted as x1, x2, x3, etc. The equation of the regression line is:

y = b0 + b1*x1 + b2*x2 + b3*x3 + ... + e

where y is the predicted value of the dependent variable, b0 is the intercept, b1, b2, b3, etc. are the coefficients that represent the change in y for a one-unit change in x1, x2, x3, etc., respectively, and e is the error term, which represents the difference between the predicted value and the actual value


Compare the throughput with the result in Section 27.3.1, where generating a single response with the same `Qwen2.5-1.5B-Instruct` model in Transformers produced about 15 tokens per second. Because the weights are read once per decoding step for the entire batch, vLLM generates several hundred tokens per second on the same GPU (about 550 in our run, more than 30 times faster), even though each individual request is not faster.

<a id="27.4.2-serving-an-openai-compatible-api"></a>

### 27.4.2 Serving an OpenAI-Compatible API

In production, the model runs as a server that receives requests over HTTP. The command `vllm serve` starts a server that implements the OpenAI API (the endpoints `/v1/chat/completions`, `/v1/completions`, and `/v1/models`). As a result, any application written for the OpenAI API, including the LangChain, CrewAI, and AutoGen code from Lecture 25, can use the self-hosted model by changing only the base URL.

First, we release the GPU memory used by the offline `LLM` object. Even if some of the memory is not released, the server can still start, since each vLLM instance uses only 40% of the GPU memory. Then, we start the server in the background with `nohup`, and save its log to the file `vllm.log`.

In [4]:
import gc, torch
del llm
gc.collect(); torch.cuda.empty_cache()

(MainProcess pid=3934) INFO 11-21 01:26:51 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


In [5]:
!nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 \
    --gpu-memory-utilization 0.4 --port 8000 > vllm.log 2>&1 &

Loading the model takes a minute or two. The next cell checks the `/health` endpoint every few seconds until the server is ready.

In [6]:
import requests, time

for _ in range(60):
    try:
        if requests.get("http://localhost:8000/health").status_code == 200:
            print("Server is ready")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    print("Server did not start. Last lines of vllm.log:")
    print(open("vllm.log").read()[-3000:])

Server is ready


The server can be called with any HTTP client, such as `curl`, similarly to the FastAPI applications in Lecture 16.

In [7]:
!curl -s http://localhost:8000/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{"model": "Qwen/Qwen2.5-1.5B-Instruct", "messages": [{"role": "user", "content": "What is MLOps? One sentence."}], "max_tokens": 60}'

{"id":"chatcmpl-833f1de80be6327b","object":"chat.completion","created":1795224570,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"MLOps stands for Machine Learning Operations and refers to the practices used to manage the entire lifecycle of machine learning projects, including data preparation, model training, deployment, monitoring, and maintenance.","refusal":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null,"routed_experts":null,"weight_versions":null}],"service_tier":null,"system_fingerprint":"vllm-0.34.0-e8c07c9c","usage":{"prompt_tokens":38,"total_tokens":77,"completion_tokens":39,"completion_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}

More conveniently, we can use the `openai` package with `base_url` set to our server. Since the server does not require authentication by default, any value can be used as the API key (in production, an API key is set with the `--api-key` option). The example below uses *streaming*, where tokens are displayed as soon as they are generated. Streaming does not reduce the total generation time, but it greatly improves the perceived responsiveness, because the user starts reading after the TTFT instead of waiting for the entire response.

In [8]:
from openai import OpenAI

local_client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
SERVED_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

stream = local_client.chat.completions.create(
    model=SERVED_MODEL,
    messages=[{"role": "user", "content": "Give three tips for writing clean Python code."}],
    max_tokens=200,
    stream=True)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

Writing clean and maintainable Python code is crucial for ensuring your programs are easy to read, understand, and modify over time. Here are three key tips to help you write cleaner Python code:

1. **Use meaningful variable names**: Choose names that clearly describe the purpose of variables, functions, and classes. Avoid using single-letter variable names unless they are part of a standard library or a built-in function. For example, instead of `x`, use `data_size` if it represents the size of data.

2. **Write modular and reusable code**: Break down complex tasks into smaller, manageable pieces (functions) that can be reused in different parts of your program. This not only makes your code more readable but also easier to test and debug. Each function should have a clear responsibility and return specific results rather than performing multiple unrelated actions.

3. **Follow PEP 8 style guide**: The Python Enhancement Proposal 8 (PEP 8) provides guidelines on how to format Python 

**Measuring the Server Under Load**

To observe continuous batching in a server, we will send 32 requests concurrently from a thread pool and measure the latency of each request and the total throughput. The response object includes the number of generated tokens in `usage.completion_tokens`.

In [9]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np

def timed_request(prompt):
    start = time.perf_counter()
    r = local_client.chat.completions.create(model=SERVED_MODEL, max_tokens=150,
                                             messages=[{"role": "user", "content": prompt}])
    return time.perf_counter() - start, r.usage.completion_tokens

# Warm-up request (the first request to the server includes one-time initialization costs)
_ = timed_request(prompts[0])

for n_concurrent in [1, 8, 32]:
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_concurrent) as pool:
        results = list(pool.map(timed_request, prompts[:n_concurrent]))
    total = time.perf_counter() - start
    latencies = [r[0] for r in results]
    tokens = sum(r[1] for r in results)
    print(f"{n_concurrent:>2} concurrent requests: mean latency {np.mean(latencies):.2f} s, "
          f"throughput {tokens / total:.0f} tokens/s")

 1 concurrent requests: mean latency 2.22 s, throughput 67 tokens/s
 8 concurrent requests: mean latency 2.31 s, throughput 463 tokens/s
32 concurrent requests: mean latency 3.43 s, throughput 1241 tokens/s


As the number of concurrent requests grows, the throughput of the server increases considerably, while the latency of each individual request increases only moderately. This illustrates the latency and throughput trade-off discussed previously.

**Deploying to Production**

In production, the vLLM server is usually run from the official Docker image (`vllm/vllm-openai`), as discussed in Lecture 17, on a cloud GPU instance or in a Kubernetes cluster. Additional components typically include an API gateway for authentication and rate limiting, a load balancer across multiple model replicas, autoscaling based on the request queue length, and monitoring of the metrics that vLLM exposes at the `/metrics` endpoint (e.g., TTFT, TPOT, and KV cache usage) with tools such as Prometheus and Grafana.

<a id="27.4.3-running-llms-locally-with-ollama"></a>

### 27.4.3 Running LLMs Locally with Ollama

For running LLMs on a laptop or a desktop computer without a data center GPU, **Ollama** provides a simple way to download and run quantized models. After installing Ollama from [ollama.com](https://ollama.com), a model is downloaded and started with a single command in the terminal:

```bash
ollama run qwen2.5:7b
```

Ollama also runs a local server with an OpenAI-compatible API at `http://localhost:11434/v1`, so the same Python code can be used:

```python
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
response = client.chat.completions.create(model="qwen2.5:7b",
                                          messages=[{"role": "user", "content": "Hello!"}])
```

Local deployment is useful for prototyping, for working with private data that must not leave the computer, and for offline use. However, Ollama and llama.cpp are designed for a single user, and they are not a replacement for vLLM or SGLang when serving many concurrent users.

<a id="27.5-evaluation-pipelines"></a>

## 27.5 Evaluation Pipelines

Deploying an LLM application takes effort as prompts are revised, models are replaced with newer or quantized versions, retrieval settings are tuned, and each change can improve some responses while making others worse. An **evaluation pipeline** is an automated process that measures the quality of an LLM application on a fixed set of test cases, so that every change can be compared to the previous version before it is deployed. Evaluation pipelines play the same role for LLM applications as unit tests and validation sets play for traditional software and machine learning models.

<a id="27.5.1-offline-and-online-evaluation"></a>

### 27.5.1 Offline and Online Evaluation

In Lecture 20, we reviewed the main methods for LLM evaluation, including benchmarks, reference-based metrics, LLM-as-a-judge, and human evaluation. For a deployed LLM application, evaluation is not a one-time comparison of models, but a continuous process that is performed both before and after each release:

- **Offline evaluation** is performed before deployment. The application is evaluated on a fixed test dataset, and the results are compared to the current production version. When the model itself changes, e.g., after quantization, public benchmarks can also be rerun with tools such as `lm-evaluation-harness` to verify that the model's general capabilities did not degrade.
- **Online evaluation** is performed after deployment. The application is monitored with user feedback (e.g., thumbs up or down), A/B tests that compare versions on real traffic, and automated checks on a sample of production requests.

Open-source tools that support building evaluation pipelines include Ragas and DeepEval (metrics for RAG and LLM applications), Inspect AI, and platforms such as LangSmith and Langfuse, which combine tracing and evaluation.

<a id="27.5.2-building-an-evaluation-pipeline"></a>

### 27.5.2 Building an Evaluation Pipeline

In this section, we build a simple evaluation pipeline for a question-answering application served by our vLLM server. The pipeline consists of the following steps:

1. A **test dataset** with questions and reference answers (often called a *golden dataset*).
2. A function that runs the **system under test** on every question, and records the responses, the latency, and the number of tokens.
3. **Reference-based metrics**: exact match and token-level F1 score.
4. An **LLM judge** that scores each response for correctness from 1 to 5.
5. A **summary report** and a **quality gate**, which decides whether the version can be deployed.

Using this pipeline, we compare two versions of the application that use different system prompts.

**Test Dataset**

In practice, the test dataset should contain several hundred examples that represent real user requests, including difficult cases and cases where the application failed in the past. For demonstration, we use a small dataset of 10 questions.

In [10]:
import pandas as pd

eval_data = pd.DataFrame([
    {"question": "What is the capital of Australia?", "reference": "Canberra"},
    {"question": "How many bits are in a byte?", "reference": "8"},
    {"question": "Which Python library is commonly used for working with DataFrames?", "reference": "pandas"},
    {"question": "What does the acronym CNN stand for in deep learning?", "reference": "Convolutional Neural Network"},
    {"question": "What is the derivative of x squared?", "reference": "2x"},
    {"question": "Which planet is known as the Red Planet?", "reference": "Mars"},
    {"question": "What is the output of len('data') in Python?", "reference": "4"},
    {"question": "Which activation function outputs max(0, x)?", "reference": "ReLU"},
    {"question": "In what year did the Apollo 11 mission land on the Moon?", "reference": "1969"},
    {"question": "What metric is the harmonic mean of precision and recall?", "reference": "F1 score"},
])
eval_data

,question,reference
0,What is the capital of Australia?,Canberra
1,How many bits are in a byte?,8
2,Which Python library is commonly used for work...,pandas
3,What does the acronym CNN stand for in deep le...,Convolutional Neural Network
4,What is the derivative of x squared?,2x
5,Which planet is known as the Red Planet?,Mars
6,What is the output of len('data') in Python?,4
7,"Which activation function outputs max(0, x)?",ReLU
8,In what year did the Apollo 11 mission land on...,1969
9,What metric is the harmonic mean of precision ...,F1 score


**Running the System Under Test**

The function `run_system()` sends each question to the served model with a given system prompt, and records the response, latency, and token count.

In [11]:
def run_system(system_prompt, data):
    records = []
    for q in data["question"]:
        start = time.perf_counter()
        r = local_client.chat.completions.create(
            model=SERVED_MODEL, temperature=0, max_tokens=150,
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": q}])
        records.append({"response": r.choices[0].message.content.strip(),
                        "latency_s": time.perf_counter() - start,
                        "output_tokens": r.usage.completion_tokens})
    return pd.concat([data.reset_index(drop=True), pd.DataFrame(records)], axis=1)

PROMPT_V1 = "You are a helpful assistant."
PROMPT_V2 = "You are a helpful assistant. Answer with only the final answer, in as few words as possible."

results_v1 = run_system(PROMPT_V1, eval_data)
results_v2 = run_system(PROMPT_V2, eval_data)
results_v2[["question", "reference", "response"]]

,question,reference,response
0,What is the capital of Australia?,Canberra,Canberra
1,How many bits are in a byte?,8,8 bits are in a byte.
2,Which Python library is commonly used for work...,pandas,Pandas
3,What does the acronym CNN stand for in deep le...,Convolutional Neural Network,Convolutional Neural Network
4,What is the derivative of x squared?,2x,2x
5,Which planet is known as the Red Planet?,Mars,Mars
6,What is the output of len('data') in Python?,4,3
7,"Which activation function outputs max(0, x)?",ReLU,ReLU (Rectified Linear Unit)
8,In what year did the Apollo 11 mission land on...,1969,1969
9,What metric is the harmonic mean of precision ...,F1 score,F1 score


Note that not all responses are correct. E.g., the model may answer that `len('data')` is 3 instead of 4. Such errors are what the evaluation metrics in the next cells should detect.

**Reference-Based Metrics**

*Exact match* checks whether the normalized response equals the reference answer, and it is a strict metric. *Token-level F1* gives partial credit for word overlap between the response and the reference. We also compute a *contains* metric, which checks whether the reference appears anywhere in the response, since verbose responses can be correct but fail the exact match.

In [12]:
import re, string
from collections import Counter

def normalize(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(response, reference):
    return float(normalize(response) == normalize(reference))

def contains(response, reference):
    return float(normalize(reference) in normalize(response))

def token_f1(response, reference):
    pred, ref = normalize(response).split(), normalize(reference).split()
    common = sum((Counter(pred) & Counter(ref)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(pred), common / len(ref)
    return 2 * precision * recall / (precision + recall)

def add_metrics(df):
    df = df.copy()
    for name, fn in [("exact_match", exact_match), ("contains", contains), ("token_f1", token_f1)]:
        df[name] = [fn(r, ref) for r, ref in zip(df["response"], df["reference"])]
    return df

results_v1, results_v2 = add_metrics(results_v1), add_metrics(results_v2)

**LLM-as-a-Judge**

For the judge, we use a larger model than the one being evaluated (`Qwen2.5-72B-Instruct` through Hugging Face Inference Providers, as in Lecture 25). The judge receives the question, the reference answer, and the response, together with a scoring rubric, and returns a score and a short justification in JSON format. Setting the temperature to 0 makes the judge's scores more consistent across runs.

In [13]:
import json
from google.colab import userdata

judge_client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=userdata.get('HF_TOKEN'))
JUDGE_MODEL = "Qwen/Qwen2.5-72B-Instruct"

JUDGE_PROMPT = '''You are evaluating the answer of an AI assistant.

Question: {question}
Reference answer: {reference}
Assistant's answer: {response}

Score the assistant's answer for correctness using this rubric:
5 - fully correct and consistent with the reference
4 - correct, with minor imprecision or unnecessary details
3 - partially correct
2 - mostly incorrect
1 - incorrect or missing

Respond only with JSON in the format {{"score": <1-5>, "reason": "<one sentence>"}}.'''

def judge(row):
    r = judge_client.chat.completions.create(
        model=JUDGE_MODEL, temperature=0, max_tokens=100,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(**row)}])
    if not r.choices:   # the provider returned an error instead of a response
        return None, f"No response from the judge: {str(r)[:200]}"
    text = r.choices[0].message.content or ""
    match = re.search(r"\{.*\}", text, re.DOTALL)   # extract the JSON object from the response
    try:
        verdict = json.loads(match.group())
        return int(verdict["score"]), verdict["reason"]
    except (AttributeError, json.JSONDecodeError, KeyError, ValueError):
        return None, "Could not parse the judge's response"

for df in [results_v1, results_v2]:
    df[["judge_score", "judge_reason"]] = [judge(row) for _, row in df.iterrows()]

results_v1[["question", "response", "judge_score", "judge_reason"]]

,question,response,judge_score,judge_reason
0,What is the capital of Australia?,The capital of Australia is Canberra.,5,The answer is fully correct and consistent wit...
1,How many bits are in a byte?,A byte is made up of 8 bits.,5,The answer is fully correct and consistent wit...
2,Which Python library is commonly used for work...,The most common Python library for working wit...,5,The answer is fully correct and consistent wit...
3,What does the acronym CNN stand for in deep le...,"In deep learning, the acronym CNN stands for C...",5,The answer is fully correct and consistent wit...
4,What is the derivative of x squared?,The derivative of \(x^2\) with respect to \(x\...,5,The answer is fully correct and consistent wit...
5,Which planet is known as the Red Planet?,The planet known as the Red Planet is Mars.,5,The answer is fully correct and consistent wit...
6,What is the output of len('data') in Python?,The output of `len('data')` in Python is 4. Th...,5,The assistant's answer is fully correct and co...
7,"Which activation function outputs max(0, x)?","The activation function that outputs `max(0, x...",5,The answer is fully correct and consistent wit...
8,In what year did the Apollo 11 mission land on...,The Apollo 11 mission landed on the Moon in 1969.,5,The answer is fully correct and consistent wit...
9,What metric is the harmonic mean of precision ...,The harmonic mean of precision and recall is c...,5,The assistant's answer is fully correct and co...


**Summary Report and Quality Gate**

Finally, we aggregate the metrics for both versions. The report shows the trade-offs between the versions in quality, latency, and cost (the number of output tokens is proportional to the cost when using a paid API).

The *quality gate* checks whether a version meets the minimum requirements for deployment. In a real project, this function would run automatically in a CI/CD pipeline (e.g., GitHub Actions) on every change to the prompts, the model, or the code, and block the deployment if the quality drops below the thresholds or below the current production version.

In [14]:
def summarize(df, name):
    return pd.Series({"exact_match": df["exact_match"].mean(),
                      "contains": df["contains"].mean(),
                      "token_f1": df["token_f1"].mean(),
                      "judge_score (1-5)": df["judge_score"].mean(),
                      "mean_latency_s": df["latency_s"].mean(),
                      "p95_latency_s": df["latency_s"].quantile(0.95),
                      "mean_output_tokens": df["output_tokens"].mean()}, name=name)

report = pd.concat([summarize(results_v1, "v1: default prompt"),
                    summarize(results_v2, "v2: concise prompt")], axis=1).round(3)
report

,v1: default prompt,v2: concise prompt
exact_match,0.000,0.700
contains,1.000,0.900
token_f1,0.150,0.773
judge_score (1-5),5.000,4.600
mean_latency_s,0.849,0.098
p95_latency_s,1.947,0.150
mean_output_tokens,55.500,4.500


In [15]:
THRESHOLDS = {"judge_score (1-5)": 4.0, "contains": 0.8, "p95_latency_s": 3.0}

def quality_gate(summary):
    passed = True
    for metric, threshold in THRESHOLDS.items():
        value = summary[metric]
        ok = value <= threshold if "latency" in metric else value >= threshold
        passed &= ok
        print(f"  {'PASS' if ok else 'FAIL'}  {metric}: {value:.3f} (threshold {threshold})")
    return passed

for version in report.columns:
    print(version)
    print("  ->", "Ready to deploy" if quality_gate(report[version]) else "Blocked", "\n")

v1: default prompt
  PASS  judge_score (1-5): 5.000 (threshold 4.0)
  PASS  contains: 1.000 (threshold 0.8)
  PASS  p95_latency_s: 1.947 (threshold 3.0)
  -> Ready to deploy 

v2: concise prompt
  PASS  judge_score (1-5): 4.600 (threshold 4.0)
  PASS  contains: 0.900 (threshold 0.8)
  PASS  p95_latency_s: 0.150 (threshold 3.0)
  -> Ready to deploy 



The concise prompt (v2) typically produces much shorter responses, which increases the exact match score, reduces latency and cost, and keeps the judge scores high. The default prompt (v1) produces verbose responses that are often correct (high *contains* and judge scores), but fail the exact match. Note also that the default prompt achieved a higher judge score, since its longer answers were correct for all questions, while the concise prompt answered `len('data')` incorrectly. Shorter responses reduce latency and cost, but can also reduce accuracy. This example shows why a single metric is rarely sufficient for evaluating LLM applications, and why the choice of metrics should reflect what matters for the users of the application.

<a id="27.6-monitoring-and-production-considerations"></a>

## 27.6 Monitoring and Production Considerations

After deployment, an LLM application must be monitored continuously, because both the user requests and the system (model versions, retrieved documents, external APIs) change over time.

**Operational Monitoring**

Operational metrics include TTFT, TPOT, end-to-end latency percentiles, throughput, request error rates, GPU utilization, KV cache usage, and queue length. Alerts are set when the metrics exceed the SLOs, and autoscaling adds or removes model replicas based on the load.

**Quality Monitoring**

Quality monitoring involves logging the prompts and responses, collecting user feedback, and running automated evaluators such as LLM judges on a sample of production traffic. Failed cases from production should be added to the test dataset, so that the evaluation pipeline improves over time.

**Cost Management**

The cost of LLM applications is driven by the number of tokens. Common strategies to reduce costs include routing simple requests to smaller models, caching responses to frequent requests, prompt caching, limiting the output length, and using quantized models for self-hosted deployments.

**Safety and Guardrails**

Deployed LLMs should include input and output guardrails for filtering harmful content, detecting prompt injection attempts, preventing the leakage of personal or confidential information, and validating structured outputs before they are used by other systems. Libraries such as NeMo Guardrails and Llama Guard provide ready-made components for these checks.

**Versioning and Reproducibility**

Each deployed version of an application should record the model name and version, the quantization, the prompts, the generation parameters, and the evaluation results. This makes it possible to reproduce issues and to roll back to the previous version when a problem is discovered, following the same principles as the DataOps practices discussed in Lecture 26.

## References

1. Efficient Memory Management for Large Language Model Serving with PagedAttention, by W. Kwon et al., available at [https://arxiv.org/abs/2309.06180](https://arxiv.org/abs/2309.06180).
2. vLLM Documentation, available at [https://docs.vllm.ai](https://docs.vllm.ai).
3. Fast Inference from Transformers via Speculative Decoding, by Y. Leviathan, M. Kalman, Y. Matias, available at [https://arxiv.org/abs/2211.17192](https://arxiv.org/abs/2211.17192).
4. LLM Inference Performance Engineering: Best Practices, Databricks, available at [https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices](https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices).
5. Hugging Face Transformers Documentation: Quantization, available at [https://huggingface.co/docs/transformers/main/en/quantization/overview](https://huggingface.co/docs/transformers/main/en/quantization/overview).
6. Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena, by L. Zheng et al., available at [https://arxiv.org/abs/2306.05685](https://arxiv.org/abs/2306.05685).
7. EleutherAI lm-evaluation-harness, available at [https://github.com/EleutherAI/lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness).
8. Ollama Documentation, available at [https://docs.ollama.com](https://docs.ollama.com).

[BACK TO TOP](#top)